# Derive model capacity from measured voltage endpoints

This notebook augments the Figure 4 lifetime-summary CSV files with `C_est`, the model-accessible capacity between the measured C/20 charge voltage endpoints. It reads raw lifetime data from the repository-local `data/cell_lifetime_data` folder.

If a local data copy is intentionally incomplete, existing `C_est` values are preserved and the repo-local fallback table `data/model_capacity_from_voltage_limits.csv` is used only when it is consistent with the measured capacity.


In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import sys
import tempfile

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start=None):
    """Locate the cloned repository so the notebook can run from any working directory."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root. Run this notebook from inside the cloned repository.")


REPO_ROOT = find_repo_root()
CODE_DIR = REPO_ROOT / "code"
NOTEBOOK_DIR = CODE_DIR / "plotting" / "f4"
SUMMARY_DIR = NOTEBOOK_DIR / "data" / "lifetime_summaries"
CAPACITY_TABLE_PATH = NOTEBOOK_DIR / "data" / "model_capacity_from_voltage_limits.csv"
RAW_DATA_ROOT = REPO_ROOT / "data" / "cell_lifetime_data"
CACHE_DIR = Path(tempfile.gettempdir()) / "f4_model_capacity_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from diagnostic_algorithm_lifetime_crate.measured_io import load_processed_voltage
from diagnostic_algorithm_lifetime_crate.schemas import schema_id_for_cell
from diagnostic_algorithm_lifetime_crate.user_functions import ocp_3

CSV_GLOB = "cell*_voltage_fit_summary.csv"
DIRECTION = "charge"
PROTOCOL_KEYWORD = "C/20"
N_Q = 4000
MAX_CAPACITY_ERROR_FOR_FALLBACK = 0.05

# Set to a list such as [11, 55] when inspecting selected cells. The publication
# workflow assumes the full raw-data repository has been placed at RAW_DATA_ROOT,
# so the default attempts all Figure 4 lifetime summaries.
TARGET_CELLS = None
WRITE_BACK = True
RECOMPUTE_FROM_RAW_WHEN_AVAILABLE = True

print("REPO_ROOT =", REPO_ROOT)
print("SUMMARY_DIR =", SUMMARY_DIR)
print("RAW_DATA_ROOT =", RAW_DATA_ROOT)
print("CAPACITY_TABLE_PATH =", CAPACITY_TABLE_PATH)


In [ ]:
def infer_cell_from_path(path_str: str):
    match = re.search(r"cell(\d{3})", str(path_str))
    return int(match.group(1)) if match else np.nan


def build_param_vector_from_row(row):
    """Return the seven-parameter vector used by the current OCP model.

    Saved-table aliases follow the internal schema: x100 = x_n,100,
    y100 = x_p,100, si_scale_a = s_V, and si_shift_b = U_off.
    """
    return np.array([
        float(row["Cn_Si"]),
        float(row["Cn_Gr"]),
        float(row["x100"]),
        float(row["Cp"]),
        float(row["y100"]),
        float(row["si_scale_a"]) if "si_scale_a" in row and pd.notna(row["si_scale_a"]) else 1.0,
        float(row["si_shift_b"]) if "si_shift_b" in row and pd.notna(row["si_shift_b"]) else 0.0,
    ], dtype=float)


def get_valid_q_range_from_row(row):
    """Compute the model-valid capacity range from anode and cathode stoichiometry limits."""
    cn_si = float(row["Cn_Si"])
    cn_gr = float(row["Cn_Gr"])
    x100 = float(row["x100"])
    cp = float(row["Cp"])
    y100 = float(row["y100"])

    cn_total = cn_si + cn_gr

    q_lo_anode = -(1.0 - x100) * cn_total
    q_hi_anode = x100 * cn_total
    q_lo_cathode = -y100 * cp
    q_hi_cathode = (1.0 - y100) * cp

    return float(max(q_lo_anode, q_lo_cathode)), float(min(q_hi_anode, q_hi_cathode))


def compute_capacity_from_measured_voltage_limits(row, vmin_meas, vmax_meas, n_q=4000):
    """Estimate model-accessible capacity between the measured voltage endpoints."""
    q_lo, q_hi = get_valid_q_range_from_row(row)
    if not np.isfinite(q_lo) or not np.isfinite(q_hi) or q_hi <= q_lo:
        return empty_capacity_result()

    q_grid = np.linspace(q_lo, q_hi, n_q)
    params = build_param_vector_from_row(row)
    voltage = np.asarray(ocp_3(params, q_grid)[0], dtype=float)

    valid = np.isfinite(q_grid) & np.isfinite(voltage)
    q_grid = q_grid[valid]
    voltage = voltage[valid]
    if len(q_grid) < 10:
        return empty_capacity_result()

    order = np.argsort(voltage)
    voltage_sorted = voltage[order]
    q_sorted = q_grid[order]
    voltage_sorted, unique_idx = np.unique(voltage_sorted, return_index=True)
    q_sorted = q_sorted[unique_idx]

    model_v_min = float(np.min(voltage_sorted))
    model_v_max = float(np.max(voltage_sorted))
    inside_window = (
        np.isfinite(vmin_meas) and np.isfinite(vmax_meas)
        and model_v_min <= vmin_meas <= model_v_max
        and model_v_min <= vmax_meas <= model_v_max
    )

    voltage_to_q = interp1d(
        voltage_sorted,
        q_sorted,
        kind="linear",
        bounds_error=False,
        fill_value=np.nan,
        assume_sorted=True,
    )

    q_at_vmax = float(voltage_to_q(vmax_meas))
    q_at_vmin = float(voltage_to_q(vmin_meas))
    c_est = abs(q_at_vmax - q_at_vmin) if np.isfinite(q_at_vmax) and np.isfinite(q_at_vmin) else np.nan

    return {
        "q_at_vmax": q_at_vmax,
        "q_at_vmin": q_at_vmin,
        "C_est": c_est,
        "model_v_min": model_v_min,
        "model_v_max": model_v_max,
        "v_window_inside_model": bool(inside_window),
    }


def empty_capacity_result():
    return {
        "q_at_vmax": np.nan,
        "q_at_vmin": np.nan,
        "C_est": np.nan,
        "model_v_min": np.nan,
        "model_v_max": np.nan,
        "v_window_inside_model": False,
    }


In [ ]:
def load_local_capacity_table(path):
    """Load derived capacity-window values used only as a repo-local fallback."""
    if not path.is_file():
        return pd.DataFrame()
    table = pd.read_csv(path)
    required = {"cell", "rpt_key"}
    if not required.issubset(table.columns):
        raise ValueError(f"{path} must contain columns: {sorted(required)}")
    return table


def ensure_capacity_columns(df):
    for column, default in {
        "q_at_vmax": np.nan,
        "q_at_vmin": np.nan,
        "C_est": np.nan,
        "model_v_min": np.nan,
        "model_v_max": np.nan,
        "v_window_inside_model": False,
        "Vmax_meas": np.nan,
        "Vmin_meas": np.nan,
    }.items():
        if column not in df.columns:
            df[column] = default
    return df


def _capacity_error_fraction(row):
    q_meas = row.get("Qcc_max_meas", np.nan)
    c_est = row.get("C_est", np.nan)
    if pd.isna(q_meas) or pd.isna(c_est) or float(q_meas) == 0:
        return np.nan
    return abs(float(c_est) - float(q_meas)) / abs(float(q_meas))


def clear_capacity_columns(df, idx):
    for col in [
        "q_at_vmax",
        "q_at_vmin",
        "C_est",
        "model_v_min",
        "model_v_max",
        "v_window_inside_model",
        "Vmax_meas",
        "Vmin_meas",
    ]:
        if col in df.columns:
            df.at[idx, col] = np.nan if col != "v_window_inside_model" else False


def fill_from_capacity_table(df, capacity_table, q_tolerance=0.02):
    """Fill missing capacity-window columns from the static fallback table.

    The table is used only when raw traces are unavailable or when an existing
    `C_est` is clearly inconsistent with the measured charge capacity. Candidate
    rows are accepted only if both the measured capacity and model capacity are
    close to the target summary row.
    """
    if capacity_table.empty:
        return df, 0

    columns_to_fill = [
        "q_at_vmax",
        "q_at_vmin",
        "C_est",
        "model_v_min",
        "model_v_max",
        "v_window_inside_model",
        "Vmax_meas",
        "Vmin_meas",
    ]
    available = [col for col in columns_to_fill if col in capacity_table.columns]
    by_key = capacity_table.set_index(["cell", "rpt_key"])

    n_filled = 0
    for idx, row in df.iterrows():
        existing_error = _capacity_error_fraction(row)
        if pd.notna(existing_error) and existing_error <= MAX_CAPACITY_ERROR_FOR_FALLBACK:
            continue
        if pd.notna(existing_error) and existing_error > MAX_CAPACITY_ERROR_FOR_FALLBACK:
            clear_capacity_columns(df, idx)
            row = df.loc[idx]

        cell = row.get("cell")
        rpt_key = row.get("rpt_key")
        q_meas = row.get("Qcc_max_meas", np.nan)
        candidate = None

        key = (cell, rpt_key)
        if key in by_key.index:
            src = by_key.loc[key]
            if isinstance(src, pd.DataFrame):
                src = src.iloc[0]
            q_src = src.get("Qcc_max_meas", np.nan)
            c_src = src.get("C_est", np.nan)
            if (
                pd.notna(q_meas)
                and pd.notna(q_src)
                and pd.notna(c_src)
                and abs(float(q_src) - float(q_meas)) <= q_tolerance
                and abs(float(c_src) - float(q_meas)) / abs(float(q_meas)) <= MAX_CAPACITY_ERROR_FOR_FALLBACK
            ):
                candidate = src

        if candidate is None and "Qcc_max_meas" in capacity_table.columns:
            same_cell = capacity_table[capacity_table["cell"] == cell].copy()
            same_cell = same_cell[pd.notna(same_cell.get("C_est", np.nan))]
            if not same_cell.empty and pd.notna(q_meas):
                distance = (same_cell["Qcc_max_meas"] - float(q_meas)).abs()
                nearest_idx = distance.idxmin()
                nearest = same_cell.loc[nearest_idx]
                nearest_error = abs(float(nearest["C_est"]) - float(q_meas)) / abs(float(q_meas))
                if float(distance.loc[nearest_idx]) <= q_tolerance and nearest_error <= MAX_CAPACITY_ERROR_FOR_FALLBACK:
                    candidate = nearest

        if candidate is None:
            continue

        for col in available:
            df.at[idx, col] = candidate[col]
        if pd.notna(df.at[idx, "C_est"]):
            n_filled += 1

    return df, n_filled


def measured_root_for_cell(cell):
    """Return the repo-local raw-data root for the cell schema."""
    schema_id = schema_id_for_cell(int(cell))
    if schema_id not in {"type1", "type2"}:
        raise ValueError(f"Cell {cell:03d} has unknown raw-data schema.")
    return RAW_DATA_ROOT / schema_id, schema_id


def fill_from_raw_voltage(df, cell):
    """Recompute C_est from repo-local raw measured voltage limits."""
    if not RECOMPUTE_FROM_RAW_WHEN_AVAILABLE:
        return df, 0

    try:
        raw_root, schema_id = measured_root_for_cell(cell)
        measured = load_processed_voltage(
            cell=cell,
            schema=schema_id,
            voltaiq_root=raw_root,
            direction=DIRECTION,
            protocol_keyword=PROTOCOL_KEYWORD,
            cache_dir=CACHE_DIR,
            use_cache=True,
            refresh_cache=False,
        )
    except Exception as exc:
        print(f"cell{cell:03d}: raw voltage unavailable in {RAW_DATA_ROOT}; preserving existing C_est and using fallback if needed ({exc})")
        return df, 0

    n_filled = 0
    for idx, row in df.iterrows():
        rpt_key = int(row["rpt_key"]) if "rpt_key" in row and pd.notna(row["rpt_key"]) else None
        if rpt_key is None or rpt_key not in measured:
            # Keep existing values when a partial example dataset does not
            # contain this RPT segment.
            continue

        voltage = np.asarray(measured[rpt_key].V_V, float)
        valid = np.isfinite(voltage)
        if np.sum(valid) < 5:
            continue

        vmax_meas = float(np.nanmax(voltage[valid]))
        vmin_meas = float(np.nanmin(voltage[valid]))
        result = compute_capacity_from_measured_voltage_limits(row, vmin_meas, vmax_meas, n_q=N_Q)
        result["Vmax_meas"] = vmax_meas
        result["Vmin_meas"] = vmin_meas
        if not np.isfinite(result.get("C_est", np.nan)):
            continue

        for col, value in result.items():
            df.at[idx, col] = value
        if pd.notna(df.at[idx, "C_est"]):
            n_filled += 1

    print(f"cell{cell:03d}: recomputed C_est from repo-local raw voltage for {n_filled} rows")
    return df, n_filled


In [ ]:
capacity_table = load_local_capacity_table(CAPACITY_TABLE_PATH)
summary_paths = sorted(SUMMARY_DIR.glob(CSV_GLOB))
if TARGET_CELLS is not None:
    target_set = {int(cell) for cell in TARGET_CELLS}
    summary_paths = [path for path in summary_paths if infer_cell_from_path(str(path)) in target_set]
print(f"Found {len(summary_paths)} summary CSVs")

all_summaries = []
for csv_path in summary_paths:
    cell = infer_cell_from_path(str(csv_path))
    if not np.isfinite(cell):
        print(f"Skip {csv_path.name}: cannot infer cell number")
        continue
    cell = int(cell)

    summary = pd.read_csv(csv_path)
    if summary.empty:
        continue

    summary = ensure_capacity_columns(summary)
    before_missing = int(summary["C_est"].isna().sum())

    summary, n_raw = fill_from_raw_voltage(summary, cell)
    summary, n_table = fill_from_capacity_table(summary, capacity_table)

    if "source_file" in summary.columns:
        summary["source_file"] = ""

    after_missing = int(summary["C_est"].isna().sum())
    print(
        f"cell{cell:03d}: missing C_est {before_missing} -> {after_missing} "
        f"(raw filled {n_raw}, table filled {n_table})"
    )

    if WRITE_BACK:
        summary.to_csv(csv_path, index=False)

    all_summaries.append(summary)

merged_summary = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
print("Merged rows:", len(merged_summary))



In [ ]:
if not merged_summary.empty:
    capacity_columns = [
        "cell",
        "rpt_key",
        "Qcc_max_meas",
        "q_at_vmax",
        "q_at_vmin",
        "C_est",
        "model_v_min",
        "model_v_max",
        "v_window_inside_model",
        "Vmax_meas",
        "Vmin_meas",
    ]
    available_columns = [col for col in capacity_columns if col in merged_summary.columns]
    capacity_out = merged_summary[available_columns].copy()
    capacity_out.to_csv(CAPACITY_TABLE_PATH, index=False)
    print("Saved:", CAPACITY_TABLE_PATH)
    display(capacity_out.head())
else:
    print("No summaries were processed.")
